# 부록 06. Middleware로 에이전트 실행 제어하기

LangChain 1.0의 Middleware 시스템은 에이전트 실행의 모든 단계를 제어할 수 있게 해줍니다.

## 학습 목표
- Middleware의 개념과 역할 이해
- 다양한 Middleware 훅 사용법
- 커스텀 Middleware 작성

## 1. 환경 설정

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

print(f"OPENAI_API_KEY: {'설정됨' if os.getenv('OPENAI_API_KEY') else '미설정'}")

## 2. Middleware란?

Middleware는 에이전트 실행의 다양한 단계에 개입할 수 있는 훅(hook)을 제공합니다.

### 주요 용도
- **모니터링**: 로깅, 분석, 디버깅
- **변환**: 프롬프트 수정, 도구 선택 변경, 출력 포맷팅
- **제어**: 재시도, 폴백, 종료 조건
- **보안**: 속도 제한, 가드레일, PII 감지

### Middleware 훅 종류
- `wrap_model_call`: 모델 호출 전후 개입
- `wrap_tool_call`: 도구 호출 전후 개입
- `dynamic_prompt`: 동적 시스템 프롬프트 생성

## 3. wrap_model_call: 모델 호출 제어

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from langchain.tools import tool

# 간단한 도구
@tool
def get_time() -> str:
    """현재 시간을 반환합니다."""
    from datetime import datetime
    return datetime.now().strftime("%H:%M:%S")

# 로깅 Middleware
@wrap_model_call
def logging_middleware(request: ModelRequest, handler) -> ModelResponse:
    """모델 호출을 로깅하는 Middleware"""
    print(f"\n📝 [LOG] 모델 호출 시작")
    print(f"   - 메시지 수: {len(request.state['messages'])}")
    print(f"   - 도구 수: {len(request.tools)}")
    
    # 실제 모델 호출
    response = handler(request)
    
    print(f"📝 [LOG] 모델 호출 완료")
    if hasattr(response, 'tool_calls') and response.tool_calls:
        print(f"   - 도구 호출: {[tc['name'] for tc in response.tool_calls]}")
    
    return response

# Middleware가 적용된 에이전트
agent = create_agent(
    model="gpt-4o-mini",
    tools=[get_time],
    system_prompt="당신은 시간을 알려주는 도우미입니다.",
    middleware=[logging_middleware]
)

print("로깅 Middleware가 적용된 에이전트 생성 완료")

In [ ]:
# 테스트
result = agent.invoke({"messages": [{"role": "user", "content": "지금 몇 시야?"}]})
print(f"\n최종 응답: {result['messages'][-1].content}")

## 4. wrap_tool_call: 도구 호출 제어

In [ ]:
from langchain.agents.middleware import wrap_tool_call, ToolCallRequest
from langchain_core.messages import ToolMessage

@tool
def divide(a: float, b: float) -> str:
    """두 숫자를 나눕니다."""
    return str(a / b)

# 도구 에러 핸들링 Middleware
@wrap_tool_call
def error_handling_middleware(request: ToolCallRequest, handler):
    """도구 에러를 처리하는 Middleware"""
    print(f"\n🔧 [TOOL] '{request.tool_call['name']}' 호출 시작")
    print(f"   - 인자: {request.tool_call['args']}")
    
    try:
        result = handler(request)
        print(f"🔧 [TOOL] 성공")
        return result
    except Exception as e:
        print(f"🔧 [TOOL] 오류 발생: {str(e)}")
        # 에러를 모델에게 알려줌
        return ToolMessage(
            content=f"도구 실행 오류: {str(e)}. 다른 방법을 시도해주세요.",
            tool_call_id=request.tool_call["id"]
        )

# 에러 핸들링이 적용된 에이전트
agent_with_error_handling = create_agent(
    model="gpt-4o-mini",
    tools=[divide],
    system_prompt="당신은 계산을 도와주는 도우미입니다.",
    middleware=[error_handling_middleware]
)

print("에러 핸들링 Middleware가 적용된 에이전트 생성 완료")

In [ ]:
# 정상 케이스
result = agent_with_error_handling.invoke({
    "messages": [{"role": "user", "content": "10 나누기 2 계산해줘"}]
})
print(f"\n최종 응답: {result['messages'][-1].content}")

In [ ]:
# 에러 케이스 (0으로 나누기)
result = agent_with_error_handling.invoke({
    "messages": [{"role": "user", "content": "10 나누기 0 계산해줘"}]
})
print(f"\n최종 응답: {result['messages'][-1].content}")

## 5. dynamic_prompt: 동적 프롬프트

In [ ]:
from langchain.agents.middleware import dynamic_prompt
from dataclasses import dataclass

# 컨텍스트 스키마
@dataclass
class UserContext:
    user_name: str
    language: str = "ko"
    expertise_level: str = "beginner"

# 동적 프롬프트 생성
@dynamic_prompt
def personalized_prompt(request: ModelRequest) -> str:
    """사용자 컨텍스트에 따라 프롬프트 생성"""
    ctx = request.runtime.context
    
    base = f"안녕하세요, {ctx.user_name}님을 돕는 어시스턴트입니다.\n"
    
    if ctx.expertise_level == "beginner":
        base += "쉽고 친절하게 설명하겠습니다. 전문 용어는 풀어서 설명해드릴게요.\n"
    elif ctx.expertise_level == "expert":
        base += "전문적인 용어를 사용하여 간결하게 답변하겠습니다.\n"
    
    if ctx.language == "en":
        base += "Please respond in English."
    else:
        base += "한국어로 답변해주세요."
    
    return base

# 동적 프롬프트가 적용된 에이전트
personalized_agent = create_agent(
    model="gpt-4o-mini",
    tools=[],
    middleware=[personalized_prompt],
    context_schema=UserContext
)

print("개인화 Middleware가 적용된 에이전트 생성 완료")

In [ ]:
# 초보자 컨텍스트로 테스트
result = personalized_agent.invoke(
    {"messages": [{"role": "user", "content": "API가 뭐야?"}]},
    context=UserContext(user_name="김철수", expertise_level="beginner")
)
print("[초보자 모드]")
print(result["messages"][-1].content)

In [ ]:
# 전문가 컨텍스트로 테스트
result = personalized_agent.invoke(
    {"messages": [{"role": "user", "content": "API가 뭐야?"}]},
    context=UserContext(user_name="박영희", expertise_level="expert")
)
print("[전문가 모드]")
print(result["messages"][-1].content)

## 6. AgentMiddleware 클래스 사용

복잡한 Middleware는 클래스로 정의하면 더 체계적으로 관리할 수 있습니다.

In [ ]:
from langchain.agents.middleware import AgentMiddleware
from typing import Any

class RateLimitMiddleware(AgentMiddleware):
    """모델 호출 횟수를 제한하는 Middleware"""
    
    def __init__(self, max_calls: int = 5):
        self.max_calls = max_calls
        self.call_count = 0
    
    def wrap_model_call(self, request: ModelRequest, handler):
        self.call_count += 1
        
        if self.call_count > self.max_calls:
            raise RuntimeError(f"모델 호출 제한 초과: {self.max_calls}회")
        
        print(f"📊 모델 호출 {self.call_count}/{self.max_calls}")
        return handler(request)
    
    def reset(self):
        """카운터 리셋"""
        self.call_count = 0

# 사용
rate_limiter = RateLimitMiddleware(max_calls=3)

rate_limited_agent = create_agent(
    model="gpt-4o-mini",
    tools=[get_time],
    system_prompt="도움이 되는 어시스턴트입니다.",
    middleware=[rate_limiter]
)

print("Rate Limit Middleware가 적용된 에이전트 생성 완료")

In [ ]:
# 여러 번 호출 테스트
rate_limiter.reset()

for i in range(4):
    try:
        result = rate_limited_agent.invoke({
            "messages": [{"role": "user", "content": f"테스트 {i+1}"}]
        })
        print(f"응답 {i+1}: 성공")
    except RuntimeError as e:
        print(f"응답 {i+1}: {e}")

## 7. 여러 Middleware 조합

In [ ]:
# 타이밍 Middleware
@wrap_model_call
def timing_middleware(request: ModelRequest, handler) -> ModelResponse:
    """실행 시간을 측정하는 Middleware"""
    import time
    start = time.time()
    response = handler(request)
    elapsed = time.time() - start
    print(f"⏱️ 모델 호출 시간: {elapsed:.2f}초")
    return response

# 메시지 카운트 Middleware
@wrap_model_call
def message_count_middleware(request: ModelRequest, handler) -> ModelResponse:
    """메시지 수를 출력하는 Middleware"""
    msg_count = len(request.state.get('messages', []))
    print(f"📨 현재 메시지 수: {msg_count}")
    return handler(request)

# 여러 Middleware 적용
multi_middleware_agent = create_agent(
    model="gpt-4o-mini",
    tools=[get_time],
    system_prompt="도움이 되는 어시스턴트입니다.",
    middleware=[
        message_count_middleware,  # 먼저 실행
        timing_middleware,         # 그 다음 실행
        logging_middleware         # 마지막 실행
    ]
)

result = multi_middleware_agent.invoke({
    "messages": [{"role": "user", "content": "현재 시간 알려줘"}]
})
print(f"\n최종 응답: {result['messages'][-1].content}")

## 8. 요약

### 이 노트북에서 배운 것

1. **Middleware 개념**
   - 에이전트 실행의 각 단계에 개입
   - 모니터링, 변환, 제어, 보안 용도

2. **주요 Middleware 훅**
   - `@wrap_model_call`: 모델 호출 전후 개입
   - `@wrap_tool_call`: 도구 호출 전후 개입
   - `@dynamic_prompt`: 동적 시스템 프롬프트

3. **AgentMiddleware 클래스**
   - 상태를 가지는 복잡한 Middleware 구현
   - 여러 훅을 하나의 클래스에서 관리

4. **Middleware 조합**
   - 리스트 순서대로 실행
   - 여러 Middleware 조합 가능

### 다음 단계
- 부록_07: Memory와 State 관리